# AiXbio — Notebook 0b: AlphaFold Fix + ESMFold Fallback

Fixed issues:
- AlphaFold now on **v6** (not v4) — use API to get the real URL
- ESMFold API check fixed: PDB starts with `HEADER`, not `ATOM`
- Local ESMFold fallback on A100 (~1.3 seq/s, 100 structures in ~80s)

In [2]:
import re, json, time, os
from pathlib import Path
import requests
from Bio import SeqIO
from tqdm.auto import tqdm

tox_reps = list(SeqIO.parse('data/toxins_clustered.fasta', 'fasta'))
struct_dir = Path('data/structures')
struct_dir.mkdir(exist_ok=True)

def get_uniprot_id(record):
    """Extract accession from sp|P12345|GENE_SPECIES header."""
    m = re.search(r'\|([A-Z0-9]+)\|', record.id)
    return m.group(1) if m else record.id.split('|')[0]


def get_alphafold_pdb_via_api(uid: str) -> str | None:
    """Query AlphaFold API for the real pdbUrl (v6, not hardcoded v4)."""
    try:
        r = requests.get(
            f'https://alphafold.ebi.ac.uk/api/prediction/{uid}', timeout=20
        )
        if r.status_code == 200:
            data = r.json()
            if data and isinstance(data, list):
                return data[0].get('pdbUrl')   # v6 URL returned dynamically
    except Exception:
        pass
    return None


def fold_with_esmfold_api(sequence: str, max_len: int = 400) -> str | None:
    """ESMFold public API. FIX: PDB starts with HEADER not ATOM."""
    seq = sequence[:max_len]
    try:
        r = requests.post(
            'https://api.esmatlas.com/foldSequence/v1/pdb/',
            data=seq,
            headers={'Content-Type': 'application/x-www-form-urlencoded'},
            timeout=120
        )
        if r.status_code == 200 and 'ATOM' in r.text[:500]:  # ✓ FIXED: was startswith('ATOM')
            return r.text
    except Exception:
        pass
    return None


print(f'Clustered toxin reps: {len(tox_reps)}')
print(f'Structures already on disk: {len(list(struct_dir.glob("*.pdb")))}')

Clustered toxin reps: 1712
Structures already on disk: 0


In [3]:
# ── Full download: AlphaFold API → ESMFold API → local ESMFold ─
# Skip if manifest already has >= TARGET structures.

TARGET = 100

try:
    with open('data/structures_manifest.json') as f:
        manifest = json.load(f)
except FileNotFoundError:
    manifest = []

if len(manifest) >= TARGET:
    print(f'Manifest already has {len(manifest)} structures. Skipping download.')
else:
    done_ids = {d['uniprot_id'] for d in manifest}
    candidates = sorted(
        [r for r in tox_reps if get_uniprot_id(r) not in done_ids],
        key=lambda r: len(r.seq)   # prefer short seqs (faster ESMFold)
    )

    for rec in tqdm(candidates, desc='Structure download'):
        if len(manifest) >= TARGET:
            break

        uid = get_uniprot_id(rec)
        seq = str(rec.seq)
        out_path = struct_dir / f'{uid}.pdb'

        if out_path.exists():
            manifest.append({'uniprot_id': uid, 'pdb_path': str(out_path),
                              'sequence': seq, 'source': 'cached'})
            continue

        pdb_content, source = None, None

        # 1. AlphaFold API (v6)
        pdb_url = get_alphafold_pdb_via_api(uid)
        if pdb_url:
            try:
                r = requests.get(pdb_url, timeout=30)
                if r.status_code == 200:
                    pdb_content, source = r.text, 'alphafold_v6'
            except Exception:
                pass

        # 2. ESMFold public API
        if pdb_content is None and len(seq) <= 400:
            pdb_content = fold_with_esmfold_api(seq)
            if pdb_content:
                source = 'esmfold_api'

        if pdb_content:
            out_path.write_text(pdb_content)
            manifest.append({'uniprot_id': uid, 'pdb_path': str(out_path),
                              'sequence': seq, 'source': source})

        time.sleep(0.25)

    # 3. Local ESMFold if still under target
    if len(manifest) < TARGET:
        print(f'Only {len(manifest)} via API. Running local ESMFold on A100...')
        import torch
        from transformers import EsmForProteinFolding, EsmTokenizer

        esm_tok = EsmTokenizer.from_pretrained('facebook/esmfold_v1')
        esm_fold = EsmForProteinFolding.from_pretrained(
            'facebook/esmfold_v1', low_cpu_mem_usage=True
        ).cuda().eval()

        done_ids = {d['uniprot_id'] for d in manifest}
        remaining = [
            (get_uniprot_id(r), str(r.seq))
            for r in sorted(tox_reps, key=lambda r: len(r.seq))
            if get_uniprot_id(r) not in done_ids and len(r.seq) <= 400
        ]

        for uid, seq in tqdm(remaining[:TARGET - len(manifest)], desc='ESMFold local'):
            out_path = struct_dir / f'{uid}.pdb'
            if out_path.exists():
                continue
            try:
                tokenized = {k: v.cuda() for k, v in
                             esm_tok([seq], return_tensors='pt',
                                     add_special_tokens=False).items()}
                with torch.no_grad():
                    output = esm_fold(**tokenized)
                pdb_str = esm_fold.output_to_pdb(output)[0]
                out_path.write_text(pdb_str)
                manifest.append({'uniprot_id': uid, 'pdb_path': str(out_path),
                                  'sequence': seq, 'source': 'esmfold_local'})
            except Exception as e:
                pass

        del esm_fold  # free VRAM for ProteinMPNN + ESM-2
        torch.cuda.empty_cache()

    with open('data/structures_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)

    af  = sum(1 for d in manifest if 'alphafold' in d.get('source', ''))
    esm = sum(1 for d in manifest if 'esmfold'   in d.get('source', ''))
    print(f'Manifest: {len(manifest)} structures ({af} AlphaFold v6, {esm} ESMFold)')

Structure download:  19%|█▉        | 323/1712 [03:21<14:27,  1.60it/s]  

Manifest: 100 structures (100 AlphaFold v6, 0 ESMFold)
